# 04 Capon and beamforming spectra from a covariance

This notebook confirms the spectral estimators behind the `capon_cycle` term. From a covariance built out of a known profile we re-estimate the height spectrum with Capon (minimum-variance) and with conventional beamforming, and compare them against the true profile. This exposes the known Capon bias that affects the power loss.

Modules exercised: `tools.sar.tomo_geometry.TomoGeometry`, the covariance and Capon arithmetic inside `PhysicsComponents.capon_cycle` (diagonal loading, matrix inversion, `1 / a^H C^-1 a`).

We reproduce the check's exact Capon estimator and show that it sharpens the spectrum and biases its integrated power relative to the input profile.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_default_dtype(torch.float32)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.titlesize" : 12,
    "axes.labelsize" : 11,
    "legend.fontsize": 9,
})

DEVICE = torch.device("cpu")


In [ ]:
from configuration.training_config import GeometryConfig
from tools.sar.tomo_geometry            import TomoGeometry
from tools.data.gaussians                import GaussianMixture

def to_check_tensor(profiles):
    t = torch.tensor(np.asarray(profiles, dtype=np.float32), device=DEVICE)
    return t.T.reshape(1, t.shape[1], 1, t.shape[0])

def gaussian_profile(x_axis, amp, mu, sigma):
    amps = np.asarray(amp,   dtype=np.float32).reshape(1, -1)
    mus  = np.asarray(mu,    dtype=np.float32).reshape(1, -1)
    sigs = np.asarray(sigma, dtype=np.float32).reshape(1, -1)
    return GaussianMixture.evaluate_batch(x_axis, amps, mus, sigs)[0]


In [ ]:
x_axis  = np.linspace(-20.0, 40.0, 160).astype(np.float32)
dx      = float(x_axis[1] - x_axis[0])
xt      = torch.tensor(x_axis, device=DEVICE)
geom    = TomoGeometry(GeometryConfig(), xt)
loading = 1e-2
floor   = 1e-3

def capon_spectrum(profile):
    prof_t   = to_check_tensor(profile[None, :])
    n_tracks = geom.steering.shape[0]
    cov      = torch.einsum("ijk,bkhw->bhwij", geom.outer, prof_t.to(geom.outer.dtype)) * dx
    trace    = torch.diagonal(cov, dim1=-2, dim2=-1).sum(dim=-1).real / n_tracks
    eye      = torch.eye(n_tracks, dtype=cov.dtype, device=cov.device)
    cov      = cov + (loading * trace.clamp(min=floor)).unsqueeze(-1).unsqueeze(-1) * eye
    inv      = torch.linalg.inv(cov)
    denom    = torch.einsum("ik,bhwij,jk->bhwk", geom.steering.conj(), inv, geom.steering).real
    spec     = (1.0 / denom.clamp(min=1e-12)).permute(0, 3, 1, 2)
    return spec[0, :, 0, 0].cpu().numpy()

def beamform_spectrum(profile):
    prof_t   = to_check_tensor(profile[None, :])
    n_tracks = geom.steering.shape[0]
    cov      = torch.einsum("ijk,bkhw->bhwij", geom.outer, prof_t.to(geom.outer.dtype)) * dx
    num      = torch.einsum("ik,bhwij,jk->bhwk", geom.steering.conj(), cov, geom.steering).real
    spec     = (num / (n_tracks ** 2)).permute(0, 3, 1, 2)
    return spec[0, :, 0, 0].cpu().numpy()


## Spectra for a single scatterer

We feed a moderately broad scatterer and overlay the input profile, the conventional beamforming spectrum (broadened by the array response), and the Capon spectrum (sharpened by minimum-variance weighting). All curves are scaled to unit peak for shape comparison.

In [ ]:
profile = gaussian_profile(x_axis, amp=1.0, mu=8.0, sigma=4.0)
cap     = capon_spectrum(profile)
bf      = beamform_spectrum(profile)

def unit_peak(z):
    return z / max(float(np.max(z)), 1e-12)

fig, ax = plt.subplots(figsize=(6.8, 3.6))
ax.plot(x_axis, unit_peak(profile), color="k",  lw=1.8, label="input profile")
ax.plot(x_axis, unit_peak(bf),      color="C1", lw=1.2, label="beamforming")
ax.plot(x_axis, unit_peak(cap),     color="C0", lw=1.2, label="Capon")
ax.set_xlabel("elevation [m]")
ax.set_ylabel("normalised spectrum")
ax.set_title("Capon vs beamforming spectral shape")
ax.legend()
plt.show()

## Integrated-power bias of the Capon re-estimate

The `capon_cycle` term builds a covariance from the predicted profile, re-estimates the spectrum with Capon, and compares the normalised result to the target. Because Capon is a biased power estimator, its integrated mass differs from the profile that generated the covariance even when the underlying scene is identical. We quantify that bias across scatterer widths.

In [ ]:
sigmas    = np.array([0.8, 1.5, 3.0, 5.0, 8.0, 12.0], dtype=np.float32)
input_pow = []
capon_pow = []

for s in sigmas:
    p = gaussian_profile(x_axis, amp=1.0, mu=8.0, sigma=float(s))
    input_pow.append(float(p.sum() * dx))
    capon_pow.append(float(capon_spectrum(p).sum() * dx))

input_pow = np.asarray(input_pow)
capon_pow = np.asarray(capon_pow)

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(sigmas, capon_pow / input_pow, "o-", color="C3")
ax.axhline(1.0, color="0.4", ls="--", lw=0.9, label="unbiased")
ax.set_xlabel("scatterer sigma [m]")
ax.set_ylabel("Capon mass / input mass")
ax.set_title("Capon integrated-power bias vs vertical extent")
ax.legend()
plt.show()

## Expected outcome

The Capon spectrum is visibly narrower than both the input profile and the beamforming spectrum. The integrated-power ratio departs from one and varies with scatterer width, demonstrating the Capon bias that makes a raw power-matching loss unreliable and motivates the normalised, shape-based formulation of the `capon_cycle` term.